# nb_03b — Gold: `fact_workforce_event` with as-of key resolution

**Module 3 (fact).** The payoff of SCD2: we attribute each pay-setting event to
the **pay grid in force on the event date**, then compute a **compa-ratio** we can
compare against *today's* re-benchmarked grid.

Resolved per event:
- `date_key` → `dim_date`
- `cost_center_key` → `dim_cost_center` (SCD1, direct)
- `worker_key` → `dim_worker` **as-of** `event_date` (SCD2 range join)
- `pay_band_key` → `dim_pay_band` **as-of** on (group, level, event_date) (SCD2 range join)

We precompute band bounds and compa-ratio at event so the model's DAX stays simple.

> **Lab notebook.** This is the fill-in-the-blank companion to the solution notebook of the same name. Each code cell only contains `# TODO` comments — use the markdown cell above each one to figure out what to build.


## Load the fact inputs

**Summary.** Loads the Silver events and the dimensions, aliasing the SCD2 validity columns so the as-of range joins that follow read clearly.


In [ ]:
# TODO: Load the Silver events and the dimensions needed to build the fact.
# - Load silver.workforce_event.
# - Load gold.dim_cost_center (key lookup for the direct SCD1 join).
# - Load gold.dim_worker, aliasing its effective_from/effective_to so the as-of range
#   join reads clearly (e.g. w_from / w_to).
# - Load gold.dim_pay_band similarly, aliasing group/level and its validity range.
# - Print how many events will be loaded.


## Build the fact with as-of key resolution

**Summary.** Resolves each event's dimension keys — cost centre directly, worker and pay band **as-of** the event date via SCD2 range joins — then precomputes the base salary, bonus, compa-ratio, and below-band flag.


In [ ]:
# TODO: Build gold.fact_workforce_event, resolving each dimension key as-of the event.
# - Derive an integer date_key (yyyyMMdd) from event_date.
# - Join dim_cost_center directly on cost_center_id (SCD1 — no date range needed).
# - Join dim_worker where employee_id matches AND event_date falls between the worker
#   version's effective_from/effective_to (an as-of SCD2 range join).
# - Join dim_pay_band where group and level match AND event_date falls within the band
#   version's validity range (another as-of SCD2 range join).
# - Compute base_salary_cad (for Hire/Promotion/Step Increment events) and bonus_cad (for
#   Performance Pay events).
# - Compute compa_ratio_at_event (base salary / band midpoint) and below_band_at_event
#   (base salary below band minimum), only where a base salary exists.
# - Select the final columns (aliasing the band bounds *_at_event) and overwrite
#   gold.fact_workforce_event. Print the row count.


## Data-quality gate — no unresolved keys should remain


## Data-quality gate — no unresolved keys

**Summary.** Counts any rows where a dimension key failed to resolve, so a broken join surfaces immediately instead of silently corrupting analysis.


In [ ]:
# TODO: Add a data-quality gate that fails loudly on unresolved keys.
# - Count how many rows have a null worker_key, pay_band_key, or cost_center_key,
#   alongside the total row count.
# - All null counts should be zero — investigate the joins if not.


## The SCD2 payoff — compa-ratio as-was vs as-is

Average compa-ratio of pay set in 2021, measured against the grid **in force then**
(as-was) — every such pay looks healthy near 1.0. But the grid has since been
re-benchmarked upward, so the same salaries sit lower against **today's** grid.
The as-is view is a model measure (Module 4); here's the as-was baseline that only
the versioned `pay_band_key` makes possible.


## The SCD2 payoff — compa-ratio as-was

**Summary.** Averages the as-was compa-ratio by the year pay was set, showing pay looked healthy against the grid in force then — the baseline only the versioned `pay_band_key` makes possible.


In [ ]:
# TODO: Show the SCD2 payoff — average as-was compa-ratio by the year pay was set.
# - Join the fact table to dim_date on date_key.
# - Keep only rows with a non-null base_salary_cad.
# - Group by the year pay was set and compute the average compa_ratio_at_event plus a
#   count of below-band pay events.
